# MedScan + Explain — train & evaluate the image classifier

Run this on a GPU runtime (Colab, or the AWS EC2 GPU instance described in `docs/aws_setup.md`).
It wraps the scripts in `src/` so the notebook stays short and the logic stays testable.

**This is a course project, not a medical device.**

In [ ]:
# If running in Colab, clone the repo first (uncomment):
# !git clone <your-github-repo-url> medscan && cd medscan
%cd ..
!pip -q install -r requirements.txt

In [ ]:
import torch
from src.config import TrainConfig
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Train EfficientNet-B0 on the pneumonia dataset
Datasets stream from the HuggingFace hub — no Kaggle login needed. First run downloads ~1–2 GB.

In [ ]:
from src.train import train
cfg = TrainConfig(dataset='pneumonia', backbone='efficientnet_b0', epochs=10)
history = train(cfg)

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'outputs/{cfg.dataset}_{cfg.backbone}_curves.png')

## 2. Evaluate (metrics, confusion matrix, ROC, baselines)

In [ ]:
from src.evaluate import evaluate
results = evaluate(str(cfg.checkpoint_path('best')), run_baselines=True)
IPyImage(filename=f'outputs/{cfg.dataset}_{cfg.backbone}_confusion_matrix.png')

## 3. (Comparison backbone) ResNet-50 — for the 'alternatives considered' table

In [ ]:
cfg_rn = TrainConfig(dataset='pneumonia', backbone='resnet50', epochs=10)
train(cfg_rn)
evaluate(str(cfg_rn.checkpoint_path('best')), run_baselines=False)

## 4. Grad-CAM on a few test images

In [ ]:
import matplotlib.pyplot as plt
from src.data import get_datasets
from src.gradcam import gradcam_overlay, describe_cam
from src.evaluate import load_checkpoint

model, cfg_loaded, _ = load_checkpoint(str(cfg.checkpoint_path('best')))
_, _, test_ds = get_datasets(cfg_loaded)
raw = test_ds.ds  # underlying HF split with PIL images

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, [0, 1, 2, 3]):
    pil = raw[i][test_ds.image_col].convert('RGB')
    overlay, cam, pred, probs = gradcam_overlay(model, cfg_loaded, pil)
    ax.imshow(overlay)
    ax.set_title(f"pred={cfg_loaded.class_names[pred]} ({probs[pred]:.0%})")
    ax.axis('off')
    print(describe_cam(cam, cfg_loaded.class_names[pred]))
plt.tight_layout(); plt.show()

## 5. (Stretch goal) HAM10000 skin lesions — 7-class

In [ ]:
cfg_skin = TrainConfig(dataset='ham10000', backbone='efficientnet_b0', epochs=15)
train(cfg_skin)
evaluate(str(cfg_skin.checkpoint_path('best')), run_baselines=True)